# DS2002 · Midterm Checkpoint

**Lab — 2026-10-23 · Fall 2026**  

---

## Lab 09 — Midterm checkpoint, week 1 of 3

**What you submit: the link to your team's main repo. Only that.** One submission per team. Do not upload separate files.

Put this notebook in the repo (the `notebooks/` folder is fine) and push it, so everything we check is in one place.

**You do not have to fill this all out.** It is a checkpoint, not a finished project. Skip any section you have not started and write "not started" next to it. What we must see is:

1. **Your charter**, committed to the repo
2. **Any progress** you have made, with code that ran and output you can point at

Start with the team setup, then work down the same eight things we listed at Monday's kickoff, as far as you have gotten. This is graded on evidence, not on intentions.

The purpose is to catch a stalled team while there is still time to unstall it. If something is not working, say so here plainly. A checkpoint that honestly reports "the API join is not working and here is the error" is worth more than one that claims progress that does not exist, and it gets you help on Monday.

### Team setup

This part is required. Fill it in once for the whole team.

You need:

- Your team name, and every member with a GitHub handle (2 to 4 people)
- A link to your team's **main repo**. This is the one link you submit.
- A link to `CHARTER.md` in that repo, the page on GitHub, not a file on someone's laptop

Nice to have, and also in your charter: this week's notebook owner, your meeting time, and the five worst problems in the data from the profile you ran on Monday. Be specific: "`unit_price` is stored as text and some values start with `$`" beats "prices are messy."

In [ ]:
team_setup = {
    'team_name': 'Storm Surge Analytics',
    'members': [
        {'name': 'Russell Fullerton', 'github': 'rfullerton42'},
        {'name': 'Todd Balwinski', 'github': 'toddbalwinski'},
    ],
    'repo_url': 'https://github.com/toddbalwinski/ds2002-midterm-project',
    'charter_url': 'https://github.com/toddbalwinski/ds2002-midterm-project/blob/main/CHARTER.md',
    'notebook_owner_this_week': 'Todd Balwinski',
    'meeting_time': 'Thursdays 8:30 PM and Sundays 6:30 PM',
    'five_worst_problems': [
        'unit_price is stored as text, and some values start with a $ sign',
        'store_id is written 60 different ways for what are really 12 stores, so it will not join to store_locations.csv',
        '884 transactions have no timestamp, and the ones that do are in mixed formats',
        'The same product appears under several SKU codes (45 SKU codes map to 20 products)',
        '130 transactions have a negative quantity, and 30 rows are exact duplicates',
    ],
}


def check_team(t, quiet=False):
    def real(v):
        return isinstance(v, str) and v.strip() != '' and not v.startswith('TODO')
    required = {
        'team name': real(t['team_name']),
        '2 to 4 members, each with a name and GitHub handle':
            2 <= len(t['members']) <= 4 and all(real(m['name']) and real(m['github']) for m in t['members']),
        'main repo link': real(t['repo_url']) and t['repo_url'].startswith('https://github.com/'),
        'charter link points at CHARTER.md': real(t['charter_url']) and t['charter_url'].endswith('CHARTER.md'),
    }
    nice = {
        'notebook owner': real(t['notebook_owner_this_week']),
        'meeting time': real(t['meeting_time']),
        'five specific problems': len(t['five_worst_problems']) >= 5 and all(real(p) for p in t['five_worst_problems']),
    }
    if not quiet:
        print('REQUIRED')
        for label, good in required.items():
            print('  PASS' if good else '  FAIL', '-', label)
        print('NICE TO HAVE')
        for label, good in nice.items():
            print('  done' if good else '  not yet', '-', label)
        print()
        print('team setup complete:', all(required.values()))
    return all(required.values())


check_team(team_setup)

Now show that the charter is actually committed. This is required. Run the next cell and leave its output in the notebook. Point `REPO_PATH` at your local clone of the team repo. If it prints a line like `a1b2c3d Add team charter`, you are done. If it prints "no commits found," the charter is not in the repo yet, so commit it and push before you submit.

In [ ]:
import subprocess
from pathlib import Path

# Point this at your local clone of the team repo (the folder that contains .git).
# Run this notebook from inside the clone (notebooks/ folder), after CHARTER.md is committed.
candidates = ['..', '../ds2002-midterm-project', 'ds2002-midterm-project', '.']
REPO_PATH = next((p for p in candidates if (Path(p) / '.git').exists()), candidates[0])
print('repo path:', REPO_PATH)

result = subprocess.run(['git', '-C', REPO_PATH, 'log', '--oneline', '--', 'CHARTER.md'],
                        capture_output=True, text=True)
if result.returncode != 0:
    print('git error:', result.stderr.strip())
print(result.stdout.strip() or 'no commits found for CHARTER.md')

### 1 — Your cleaning pipeline as functions

Every cleaning step is a function that prints its row count before and after and writes an entry to the decision log (item 2). The first cell loads the data and defines the functions. The second cell defines the SKU mapping and runs the whole pipeline.

Decision on negative `quantity`: we **keep** them and treat them as returns. They are reported separately as `returned_units`, and `net_units` is gross minus returns. Rows with no usable date are left out of the analysis table (not out of the cleaned table).

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from pathlib import Path
from IPython.display import display


def find_data_dir():
    for folder in (Path('data'), Path('../data'), Path('/content/data'), Path('.')):
        if (folder / 'walmart_transactions.csv').exists():
            return folder
    raise FileNotFoundError('walmart_transactions.csv not found. Run inside the repo, or upload the data/ folder.')


DATA_DIR = find_data_dir()
transactions = pd.read_csv(DATA_DIR / 'walmart_transactions.csv')
stores = pd.read_csv(DATA_DIR / 'store_locations.csv')
print('data folder:', DATA_DIR)
print('transactions:', transactions.shape, '| stores:', stores.shape)

# ---- decision log: every cleaning function adds a row here -------------------------------
decision_log = []


def log_decision(step, decision, affected, detail=''):
    decision_log.append({'step': step, 'decision': decision,
                         'rows_or_values_affected': affected, 'detail': detail})


CATEGORY_MAP = {
    'Breakfast Foods': 'Breakfast',
    'Emergency Supplies': 'Emergency',
    'Alcoholic Beverages': 'Alcohol',
    'Games & Entertainment': 'Entertainment',
    'Groceries': 'Grocery',
    'Household Essentials': 'Household',
    'Health & Wellness': 'Health',
    'Baby Products': 'Baby',
}


def normalize_store_id(series):
    return (series.astype(str).str.strip().str.upper()
            .str.replace('_', '-', regex=False)
            .str.replace(r'^FL-?(\d+)$', r'FL-\1', regex=True))


def drop_duplicate_rows(df):
    before = len(df)
    out = df.drop_duplicates().copy()
    removed = before - len(out)
    print(f'drop_duplicate_rows: {before:,} -> {len(out):,} rows ({removed} removed)')
    log_decision('duplicates', 'Dropped exact duplicate rows', removed, f'{before:,} -> {len(out):,} rows')
    return out


def clean_store_ids(df):
    out = df.copy()
    before_unique = out['store_id'].nunique()
    cleaned = normalize_store_id(out['store_id'])
    changed = int((out['store_id'].astype(str) != cleaned).sum())
    out['store_id'] = cleaned
    print(f'clean_store_ids: {before_unique} distinct ids -> {out["store_id"].nunique()} ({len(out):,} rows kept)')
    log_decision('store_id', 'Standardized store_id: strip, uppercase, "_" to "-", FL-### format', changed,
                 f'{before_unique} distinct ids -> {out["store_id"].nunique()}')
    return out


def parse_timestamps(df):
    out = df.copy()
    missing_before = int(out['timestamp'].isna().sum())
    out['timestamp'] = pd.to_datetime(out['timestamp'], format='mixed', errors='coerce')
    out['date'] = out['timestamp'].dt.normalize()
    missing_after = int(out['timestamp'].isna().sum())
    print(f'parse_timestamps: {len(out):,} rows kept; missing {missing_before} -> {missing_after}; '
          f'range {out["date"].min().date()} to {out["date"].max().date()}')
    log_decision('timestamp', 'Parsed mixed-format timestamps, derived date', missing_after,
                 f'{missing_before} missing before parsing, {missing_after - missing_before} unparseable; '
                 f'rows with no date are kept here but left out of the analysis table')
    return out


def clean_numbers(df):
    out = df.copy()
    price_text = out['unit_price'].astype(str)
    with_dollar = int(price_text.str.contains('$', regex=False).sum())
    out['quantity'] = pd.to_numeric(out['quantity'], errors='coerce')
    out['unit_price'] = pd.to_numeric(price_text.str.replace('$', '', regex=False).str.strip(), errors='coerce')
    bad_price = int(out['unit_price'].isna().sum())
    negative_qty = int((out['quantity'] < 0).sum())
    missing_qty = int(out['quantity'].isna().sum())
    out['revenue'] = out['quantity'] * out['unit_price']
    print(f'clean_numbers: {len(out):,} rows kept; {with_dollar} prices had a $; '
          f'{bad_price} prices missing or unparseable; {negative_qty} negative quantities; {missing_qty} missing quantities')
    log_decision('unit_price', 'Removed $ and converted unit_price to numeric', with_dollar,
                 f'{bad_price} values still missing or unparseable afterwards')
    log_decision('quantity', 'Converted quantity to numeric; kept negative quantities as returns', negative_qty,
                 f'{missing_qty} missing quantities (excluded from the analysis table)')
    return out


def clean_categories(df):
    out = df.copy()
    before = out['category'].nunique()
    cleaned = out['category'].astype(str).str.strip().str.title().replace(CATEGORY_MAP)
    changed = int((out['category'].astype(str) != cleaned).sum())
    out['category'] = cleaned
    print(f'clean_categories: {before} distinct categories -> {out["category"].nunique()} ({len(out):,} rows kept)')
    log_decision('category', 'Standardized category spelling and mapped 8 long names to short names', changed,
                 f'{before} distinct -> {out["category"].nunique()}')
    return out


def clean_store_file(raw):
    out = raw.copy()
    out['store_id'] = normalize_store_id(out['store_id'])
    before = len(out)
    out = out.drop_duplicates(subset=['store_id']).copy()
    missing_coords = int(out[['latitude', 'longitude']].isna().any(axis=1).sum())
    print(f'clean_store_file: {before} -> {len(out)} stores; {missing_coords} store(s) missing coordinates')
    log_decision('store file', 'Cleaned store_id and dropped duplicate stores (first row kept)', before - len(out),
                 f'{missing_coords} store(s) still missing latitude/longitude')
    return out


def attach_store_info(df, stores_clean):
    out = df.merge(stores_clean[['store_id', 'city', 'region', 'latitude', 'longitude']],
                   on='store_id', how='left', validate='many_to_one')
    unmatched = int(out['city'].isna().sum())
    print(f'attach_store_info: {len(df):,} -> {len(out):,} rows; {unmatched} without a store match')
    log_decision('store join', 'Joined transactions to the cleaned store file', unmatched,
                 'rows with no store match (0 means every transaction joined)')
    return out

In [ ]:
# ---- SKU consolidation: 20 product families, 45 SKU codes ---------------------------------
SKU_GROUPS = {
    'Strawberry Pop-Tarts': ['PT-12', 'SKU#459812', 'POPTART-STRAW', 'PT_STRAWBERRY'],
    'Bottled Water': ['WTR-24', 'WATER24PK', 'WTR_24PK'],
    'AA Batteries': ['BATT-100', 'BAT-AA4', 'BATT_AA_4PK'],
    'Flashlights': ['FLASH-01', 'LED-FLASH'],
    'Light Beer': ['BEER-24', 'LTBEER24'],
    'Monopoly Board Game': ['BG-MONO', 'MONOPOLY-01', 'GAME-MONO'],
    'Playing Cards': ['CARDS-01', 'PLAYCARD-STD'],
    'Streaming Device': ['STREAM-01', 'ROKU-STD'],
    'Canned Soup': ['SOUP-CAN', 'CNSOUP-01'],
    'White Bread': ['BREAD-WH', 'WHITEBREAD'],
    'Peanut Butter': ['PB-16', 'PNUTBTR-16'],
    'Emergency Candles': ['CNDL-EM', 'CANDLE-EMRG'],
    'First Aid Kit': ['FAID-01', 'FIRSTAID-KIT'],
    'Portable Charger': ['CHRG-PORT', 'PORTCHARGER'],
    'Chips Variety Pack': ['CHIPS-VP', 'CHIPVARIETY'],
    'Paper Towels': ['PTOWEL-6', 'PAPERTOWEL6'],
    'Toilet Paper': ['TP-12', 'TOILETPAPER12'],
    'Diapers': ['DIAP-S3', 'DIAPERS-3'],
    'Baby Formula': ['BFORM-01', 'BABYFORMULA'],
    'Heavy Duty Tarp': ['TARP-HD', 'HDTARP-01'],
}
SKU_MAP = {sku: family for family, skus in SKU_GROUPS.items() for sku in skus}


def add_product_family(df):
    out = df.copy()
    out['sku'] = out['sku'].astype(str).str.strip().str.upper()
    sku_codes = out['sku'].nunique()
    out['product_family'] = out['sku'].map(SKU_MAP)
    unmapped = int(out['product_family'].isna().sum())
    print(f'add_product_family: {len(out):,} rows kept; {sku_codes} SKU codes -> '
          f'{out["product_family"].nunique()} product families; {unmapped} unmapped rows')
    log_decision('sku', 'Mapped SKU codes to canonical product families (SKU_GROUPS)', int(len(out) - unmapped),
                 f'{sku_codes} SKU codes -> {out["product_family"].nunique()} families; {unmapped} rows unmapped')
    return out


def build_analysis_sales(df):
    keep = df['date'].notna() & df['quantity'].notna() & df['product_family'].notna()
    out = df.loc[keep].copy()
    out['gross_units'] = out['quantity'].clip(lower=0)
    out['returned_units'] = -out['quantity'].clip(upper=0)
    out['net_units'] = out['quantity']
    excluded = len(df) - len(out)
    print(f'build_analysis_sales: {len(df):,} -> {len(out):,} rows ({excluded} excluded: no date, quantity or product)')
    log_decision('analysis table', 'Excluded rows with no date, quantity or product from the analysis table', excluded,
                 f'{len(df):,} -> {len(out):,} rows; gross, returned and net units added')
    return out


def run_cleaning_pipeline(raw_transactions, raw_stores):
    print('raw transaction rows:', f'{len(raw_transactions):,}')
    stores_clean = clean_store_file(raw_stores)
    df = drop_duplicate_rows(raw_transactions)
    df = clean_store_ids(df)
    df = parse_timestamps(df)
    df = clean_numbers(df)
    df = clean_categories(df)
    df = attach_store_info(df, stores_clean)
    df = add_product_family(df)
    analysis = build_analysis_sales(df)
    return stores_clean, df, analysis


decision_log.clear()          # so Restart and Run All never double-logs
clean_stores, clean_transactions, analysis_sales = run_cleaning_pipeline(transactions, stores)

print()
print('ROW COUNTS')
print('  raw transactions:       ', f'{len(transactions):,}')
print('  after cleaning:         ', f'{len(clean_transactions):,}')
print('  analysis-ready:         ', f'{len(analysis_sales):,}')
print('  gross / returned / net units:',
      int(analysis_sales['gross_units'].sum()), '/', int(analysis_sales['returned_units'].sum()), '/',
      int(analysis_sales['net_units'].sum()))

### 2 — Your decision log

Every decision comes from the cleaning functions above, so the counts are the ones the code produced on this run.

**Disagreements:** none recorded so far.

In [ ]:
pd.set_option('display.max_colwidth', None)
decision_df = pd.DataFrame(decision_log)
decision_df.index = range(1, len(decision_df) + 1)
display(decision_df)
print(len(decision_df), 'decisions logged')

### 3 — SKU consolidation progress

The mapping table (`SKU_GROUPS` in the cleaning section above) covers all 20 products in the catalog. The cell below reports the counts and shows the Pop-Tarts family. Not yet checked: whether every SKU in the inventory database maps to a family (that belongs to Q5).

In [ ]:
raw_skus = transactions['sku'].astype(str).str.strip().str.upper()
print('distinct SKU codes in transactions:', raw_skus.nunique())
print('canonical products (product families):', clean_transactions['product_family'].nunique())
print('unmapped transaction rows:', int(clean_transactions['product_family'].isna().sum()))

sizes = {family: len(skus) for family, skus in SKU_GROUPS.items()}
largest = max(sizes, key=sizes.get)
print(f'largest consolidated group: {largest} ({sizes[largest]} SKU codes)')

print()
print('Mapping for', largest, '(transactions per SKU code):')
poptarts = (clean_transactions.loc[clean_transactions['product_family'] == largest, 'sku']
            .value_counts().rename_axis('sku').reset_index(name='transactions'))
display(poptarts)

print('Top 5 product families by transaction count:')
display(clean_transactions['product_family'].value_counts().head(5).rename_axis('product_family').reset_index(name='transactions'))

### 4 — One working weather API call

Source: Open-Meteo historical archive (`https://archive-api.open-meteo.com/v1/archive`, no key needed). The call retries up to 3 times with backoff on 429/5xx and on network errors, raises a clear error if the response has no `daily` data, and adds a `source` column. If it fails, `weather` is set to `None` and no weather values are invented.

**Timezone:** the request asks for `timezone=America/New_York`, so the daily weather rows are Eastern calendar dates. The sales timestamps parse as plain datetimes, and we reduce them to a date. We are assuming they are Eastern local time. The cell prints the sales timestamp dtype, which shows whether they carry a timezone; this assumption is not verified yet.

**Join:** daily sales are joined to daily weather on `date`, one-to-one, with an assertion that the row count does not change and a printed list of unmatched dates.

**Limit:** one point (Orlando, 28.54, -81.38) is used for all stores. Per-store weather is a later step for Q3 and Q4.

In [ ]:
WEATHER_URL = 'https://archive-api.open-meteo.com/v1/archive'
RETRYABLE = {429, 500, 502, 503, 504}


def fetch_daily_weather(start_date='2024-09-01', end_date='2024-09-30',
                        latitude=28.54, longitude=-81.38, attempts=3):
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'start_date': start_date,
        'end_date': end_date,
        'daily': 'precipitation_sum,wind_speed_10m_max,pressure_msl_mean',
        'timezone': 'America/New_York',
    }
    for attempt in range(attempts):
        try:
            response = requests.get(WEATHER_URL, params=params, timeout=25)
            if response.status_code in RETRYABLE and attempt < attempts - 1:
                time.sleep(2 ** attempt)
                continue
            response.raise_for_status()
            payload = response.json()
            if 'daily' not in payload:
                raise ValueError(f'Weather response missing daily data: {str(payload)[:300]}')
            result = pd.DataFrame(payload['daily']).rename(columns={'time': 'date'})
            result['date'] = pd.to_datetime(result['date'], errors='raise').dt.normalize()
            result['source'] = 'Open-Meteo historical archive'
            result['timezone'] = payload.get('timezone', 'America/New_York')
            if result['date'].duplicated().any():
                raise ValueError('Weather response has repeated dates')
            return result
        except requests.RequestException:
            if attempt == attempts - 1:
                raise
            time.sleep(2 ** attempt)


try:
    weather = fetch_daily_weather()
    print('Weather source:', WEATHER_URL)
    print('Weather timezone:', weather['timezone'].iloc[0])
    print('Weather records:', len(weather))
    display(weather.head())
except (requests.RequestException, ValueError) as error:
    weather = None
    print('Historical weather request failed:', error)
    print('No fabricated or fallback weather values were substituted.')

# ---- daily sales and the checked join ----------------------------------------------------
print('sales timestamp dtype:', clean_transactions['timestamp'].dtype, '(a dtype like datetime64[ns] with no timezone after it means none is attached)')

sales_daily = (
    analysis_sales.groupby('date', as_index=False)
    .agg(transaction_count=('quantity', 'size'),
         gross_units=('gross_units', 'sum'),
         returned_units=('returned_units', 'sum'),
         net_units=('net_units', 'sum'))
)

if weather is not None:
    sales_weather = sales_daily.merge(weather, on='date', how='left', validate='one_to_one', indicator=True)
    assert len(sales_weather) == len(sales_daily), 'Weather join changed the number of sales days'
    unmatched_dates = sales_weather.loc[sales_weather['_merge'] != 'both', 'date']
    print('Sales days:', len(sales_daily), '| weather days:', len(weather), '| joined rows:', len(sales_weather))
    print('Unmatched sales dates:', unmatched_dates.dt.strftime('%Y-%m-%d').tolist())
    sales_weather = sales_weather.drop(columns='_merge')
    display(sales_weather.head())
else:
    sales_weather = None
    print('Weather join skipped until the real API request succeeds.')

### 5 — One chart

Daily transaction count across September 2024, built from the analysis-ready table (returns are counted as transactions). The cell below also lists any day that is zero or far from the median, so we can tell a cleaning bug from real demand before Q1.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(sales_daily['date'], sales_daily['transaction_count'], marker='o')
ax.set_title('Daily transaction count, Florida stores, September 2024')
ax.set_xlabel('Date')
ax.set_ylabel('Transactions per day')
ax.set_ylim(bottom=0)
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.show()

median = sales_daily['transaction_count'].median()
odd = sales_daily[(sales_daily['transaction_count'] == 0)
                  | (sales_daily['transaction_count'] > 1.5 * median)
                  | (sales_daily['transaction_count'] < 0.5 * median)]
print('days in range:', len(sales_daily), '(September has 30)')
print('median transactions per day:', median)
print('days that are zero or far from the median:')
print(odd[['date', 'transaction_count']].to_string(index=False) if len(odd) else '  none')

**What we see so far:** in the first five rows of the joined table, daily counts sit around 570 to 590 and then jump to 809 on Sept 5. The cell above lists any days that stand out on the full month. Before Q1 we will check each one to decide whether it is real demand or a pipeline problem.

### 6 — Self-check

Fill in what you have and leave the rest. **The required items are the team setup and the charter.** Everything else is progress we want to see, not a pass or fail. The checks are shape checks; they cannot verify your analysis is right, only that you have gotten far enough to have one.

In [ ]:
required = {
    'team_setup_complete': check_team(team_setup, quiet=True),   # from the team setup section
    'charter_committed': bool(result.stdout.strip()),           # True only if CHARTER.md shows in the git log above
}

progress = {
    'rows_after_cleaning': len(clean_transactions),
    'decisions_logged': len(decision_log),
    'canonical_products': int(clean_transactions['product_family'].nunique()),
    'weather_api_rows': len(weather) if weather is not None else None,
    'notebook_runs_clean': False,       # set to True only after Restart and Run All finishes with no errors
}


def is_done(value):
    return not (value is None or value is False
                or (isinstance(value, str) and value.startswith('TODO')))


def check(required, progress):
    print('REQUIRED')
    for label, value in required.items():
        print('  PASS' if is_done(value) else '  FAIL', '-', label, '=', value)
    print('PROGRESS (fill in what you have)')
    for label, value in progress.items():
        print('  done' if is_done(value) else '  not yet', '-', label, '=', value)
    ok = all(is_done(v) for v in required.values())
    print()
    print('required items complete:', ok)
    print('progress items done:', sum(is_done(v) for v in progress.values()), 'of', len(progress))
    return ok


check(required, progress)

### 7 — Risk and plan

Two questions, answered concretely.

**a)** What is the single most likely thing to go wrong in the next two weeks? Name a specific technical or team problem, not "running out of time."

**b)** What are you doing about it this week? Name who is doing it and by when.

**Biggest risk:** 884 transactions (about 3.5% of the 25,532 after removing duplicates) have no timestamp, and we leave all of them out of the analysis table. If they are not spread evenly across stores, categories and days, they could bias the Q1 baseline and the size of the surge. We have not checked this yet.

**Plan, with an owner and a date:** Russell Fullerton compares the share of missing-date rows by store and category against all rows, and checks whether any other column can recover their dates. The result and the decision (drop, keep or impute) go in the decision log. Done by the Thursday, Oct 29 team meeting.

**Secondary risks:** (1) The weather call uses a single point (Orlando) for all 12 stores, which limits Q3 and Q4. Todd Balwinski tests per-store coordinates by Oct 29. (2) The sales timestamps are assumed to be Eastern time and this is not verified. Todd checks it against the weather dates by Oct 29.

### 8 — Where each question stands

These owners match the question owners table in `CHARTER.md`. No question has been answered yet, on purpose: we cleaned first.

| Question | Status | Who owns it |
|---|---|---|
| Q1 demand surge | not started. Dates, categories and product families are clean. Still needed: a baseline window and the surge calculation. | Russell Fullerton |
| Q2 SKU consolidation | in progress. The mapping table is built (45 SKU codes to 20 products, 0 unmapped transaction rows). Still needed: the before and after comparison. | Russell Fullerton |
| Q3 weather correlation | in progress. The API call works and the daily join passes its checks. Still needed: confirm the timezone, then the correlation and lag. | Todd Balwinski |
| Q4 store and geography | not started. Store ids are clean and join with 0 unmatched. Still needed: handle the store missing coordinates, define distance from the storm. | Todd Balwinski |
| Q5 signal or artifact | not started. The inventory database is not part of this notebook yet. | Russell Fullerton |